# Step 5 · Estimate pathogen publications and calculate the PII

For each accepted pathogen–host pair, estimate disease-asserting publications as `(positive sampled abstracts / selected sample records) × pair-search records`. The denominator is the selected PMID sample (capped at 200 per pair). Every available abstract in the sample must be screened; the positive sample rate supplies the publication estimate. Divide that estimate by the organism-only PubMed total to calculate the pair-specific pathogenicity fraction. The displayed PII applies `log(1 + fraction)` and divides by the maximum transformed fraction within the pathogen group, producing a 0–1 group-normalized index. Count-only PubMed searches run only for pathogens accepted in at least one selected host.


## Step 1 · Load accepted pathogen–host pairs

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, ncbi_contact_email
from pathogen_atlas_v4.runs import resolve_run
from pathogen_atlas_v4.pathogen_screening import build_pathogen_screen_tasks
from pathogen_atlas_v4.decision_runner import decision_cache_key_for_task
from pathogen_atlas_v4.pubmed_client import count_accepted_pathogen_publications, ncbi_api_key, literature_cache_key
from pathogen_atlas_v4.interest_index import build_pathogenic_interest_index
CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(PROJECT_ROOT)
RUN_MODE = str(CONFIG['run'].get('selection_mode', '')).casefold()
IS_FULL_RUN = RUN_MODE == 'full'
ROOT, LOG, LOG_PATH = configure_notebook('05_estimate_pathogen_publications', RUN_ID)
STAGING_TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
ALIASES = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet', columns=['pathogen_id','name'])
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)), known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)), host_group_ids={k:g.host_id.astype(str).tolist() for k,g in HOSTS.groupby('host_group')}, pathogen_group_ids={k:g.pathogen_id.astype(str).tolist() for k,g in PATHOGENS.groupby('pathogen_group')})
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
                                  pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
                                  taxonomy_snapshot=STAGING_TAXONOMY_ROOT.parent.name)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
PAIR_SEARCHES_PATH = CACHE_DIR / 'tables/pair_searches.parquet'
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
SCREEN_LEDGER_PATH = ROOT / 'data/v4/cache/decisions/pathogen_screen_ledger.parquet'
COUNT_PATH = CACHE_DIR / 'tables/accepted_pathogen_publication_counts.parquet'
PII_PATH = RUN_DIR / 'release/pathogenic_interest_index.parquet'
if not PAIR_SELECTION_PATH.exists():
    raise FileNotFoundError(f'Run 02b and complete pathogen screening first: {PAIR_SELECTION_PATH}')
pair_selection = pd.read_parquet(PAIR_SELECTION_PATH)
if IS_FULL_RUN:
    if len(pair_selection) != len(resolved.pairs) or pair_selection.duplicated(['pathogen_id', 'host_id']).any():
        raise RuntimeError(f'Full-run pair selection does not cover all {len(resolved.pairs):,} configured pairs. Rerun 02b to completion.')
    ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
    if not PAIR_SEARCHES_PATH.exists() or not SAMPLE_MATCHES_PATH.exists() or not ARTICLES_PATH.exists():
        raise FileNotFoundError('Full-run pair searches, screening samples, and article cache must exist before PII release.')
    pair_searches = pd.read_parquet(PAIR_SEARCHES_PATH)
    selected_pair_keys = set((str(pathogen_id), str(host_id)) for pathogen_id, host_id in resolved.pairs)
    pair_search_index = pd.MultiIndex.from_frame(pair_searches[['pathogen_id', 'host_id']].astype(str))
    pair_searches = pair_searches.loc[pair_search_index.isin(selected_pair_keys)].copy()
    acquired_pair_keys = set(map(tuple, pair_searches.loc[pair_searches.status.astype(str).eq('completed'), ['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy()))
    missing_pair_searches = selected_pair_keys - acquired_pair_keys
    if missing_pair_searches:
        raise RuntimeError(f'Full-run pair acquisition is incomplete: {len(missing_pair_searches):,} configured pairs lack a completed PubMed search. Rerun 03_acquire_pair_pubmed_records.')
    sample_matches = pd.read_parquet(SAMPLE_MATCHES_PATH)
    if not sample_matches.empty:
        sample_pair_index = pd.MultiIndex.from_frame(sample_matches[['pathogen_id', 'host_id']].astype(str))
        sample_matches = sample_matches.loc[sample_pair_index.isin(selected_pair_keys)].copy()
    PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
    articles = pd.read_parquet(ARTICLES_PATH)
    screen_tasks = build_pathogen_screen_tasks(
        sample_matches, articles, PATHOGENS, HOSTS, PROMPTS, run_id=RUN_ID,
        run_config_version=str(CONFIG['run']['config_version']),
        model_version=str(CONFIG['decisions'].get('model_version', 'typesafe/jev-1.14')))
    required_task_keys = {decision_cache_key_for_task(task) for task in screen_tasks}
    search_pair_keys = set(map(tuple, pair_searches.loc[pd.to_numeric(pair_searches.search_count, errors='coerce').fillna(0).gt(0), ['pathogen_id', 'host_id']].astype(str).to_numpy()))
    expected_sample_pairs = selected_pair_keys & search_pair_keys
    prepared_sample_pairs = set(map(tuple, sample_matches[['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy())) if not sample_matches.empty else set()
    if expected_sample_pairs - prepared_sample_pairs:
        raise RuntimeError(f'Full-run sample cache misses {len(expected_sample_pairs - prepared_sample_pairs):,} PubMed-hit pairs. Rerun 02b sample preparation.')
    ledger = pd.read_parquet(SCREEN_LEDGER_PATH) if SCREEN_LEDGER_PATH.exists() else pd.DataFrame()
    if not ledger.empty and {'decision_kind', 'cache_key'}.issubset(ledger.columns):
        ledger = ledger.loc[ledger.decision_kind.astype(str).eq('pathogen_screen')
                            & ledger.cache_key.astype(str).isin(required_task_keys)]
    else:
        ledger = pd.DataFrame()
    completed_keys = set(ledger.loc[ledger.status.astype(str).eq('completed'), 'cache_key'].astype(str)) if not ledger.empty else set()
    missing_task_keys = required_task_keys - completed_keys
    if missing_task_keys:
        raise RuntimeError(f'Full-run screening is incomplete: {len(missing_task_keys):,} current abstract tasks remain unscreened. Finish 02b before PII release.')
    selected_pairs = set(map(tuple, pair_selection[['pathogen_id', 'host_id']].astype(str).to_numpy()))
    expected_pairs = set((str(pathogen_id), str(host_id)) for pathogen_id, host_id in resolved.pairs)
    if selected_pairs != expected_pairs:
        raise RuntimeError('Full-run pair selection does not match the configured pathogen–host pairs.')
    if pair_selection.selection_status.astype(str).eq('incomplete').any():
        raise RuntimeError('Full-run pair selection still contains incomplete pairs; rerun 02b before PII release.')
accepted_pairs = pair_selection.loc[pair_selection.selection_status.eq('accepted') & pair_selection.selected_as_pathogen_for_host.fillna(False).astype(bool)].copy()
accepted_pathogen_ids = sorted(accepted_pairs.pathogen_id.astype(str).unique())
print('Accepted pathogen–host pairs:', len(accepted_pairs), '| unique accepted pathogen IDs:', len(accepted_pathogen_ids))
print('Only these pathogen IDs are sent to the PubMed total-publication count query.')
LOG.info('PII preflight: accepted_pairs=%d accepted_pathogens=%d', len(accepted_pairs), len(accepted_pathogen_ids))

2026-11-02 17:36 INFO Notebook session started at 2026-11-02T17:36:42.743949+00:00
2026-11-02 17:36 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-02 17:36 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02c_pathogenic_interest_index.log


RuntimeError: Full-run pair selection still contains incomplete pairs; rerun 02b before PII release.

## Step 2 · Count total organism publications for accepted pathogens only

These count-only searches provide the denominator. Each pathogen accepted for one or more hosts is queried once.


In [ ]:
RUN_PUBMED_COUNT = IS_FULL_RUN  # Full mode completes missing accepted-pathogen counts; cached work is resumed.
if RUN_PUBMED_COUNT:
    NCBI_EMAIL = ncbi_contact_email(ROOT)
    NCBI_API_KEY = ncbi_api_key()
    counts = count_accepted_pathogen_publications(
        accepted_pathogen_ids, PATHOGENS, ALIASES, CACHE_DIR,
        query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
        email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG)
    display(counts.head(20))
else:
    counts = pd.read_parquet(COUNT_PATH) if COUNT_PATH.exists() else pd.DataFrame()
    print('PubMed count queries skipped. Set RUN_PUBMED_COUNT=True to search only the accepted pathogen set.')
    print('Cached accepted-pathogen counts available:', len(counts))

## Step 3 · Extrapolate pair evidence and calculate the PII


In [ ]:
if accepted_pathogen_ids and counts.empty:
    raise FileNotFoundError(f'Run accepted-pathogen PubMed count queries first: {COUNT_PATH}')
pii = build_pathogenic_interest_index(counts, pair_selection, PATHOGENS)
RUN_DIR.joinpath('release').mkdir(parents=True, exist_ok=True)
pii.to_parquet(PII_PATH, index=False)
print('Saved:', PII_PATH)
if pii.empty:
    print('No accepted pathogen–host pairs; no PII rows are emitted.')
else:
    display(pii.sort_values('pathogenic_interest_index', ascending=False).head(25))
LOG.info('PII table saved: accepted_pairs=%d accepted_pathogens=%d rows=%d path=%s',
         len(accepted_pairs), len(accepted_pathogen_ids), len(pii), PII_PATH)


## Interpretation

The sample numerator is abstracts that explicitly support disease caused by this pathogen in the selected host. Divide that count by the full selected PMID sample, then extrapolate across all pair-search records. Every available abstract must be screened before the estimate is produced. Divide the estimated count by the organism-only PubMed total to get the pair-specific pathogenicity fraction. The final PII is `log(1 + fraction)` divided by the maximum transformed fraction within the pathogen group, so it ranges from 0 to 1 and is normalized separately by group. The output retains the selected sample size, screened-abstract count, pair-search count, estimated pathogenicity-positive publications, and organism-publication denominator for audit.
